In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Circuits remote transpilieren mit dem Qiskit Transpiler Service

> **Danger:** Ab dem 18. Juli 2025 wird der Service migriert, um die neue IBM Quantum&reg; Platform zu unterstützen, und ist derzeit nicht verfügbar. Für KI-Passes kannst du den [lokalen Modus](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud) verwenden.
> 
>     Der Service ist eine Beta-Version und kann sich noch ändern.
>     Wenn du Feedback hast oder das Entwicklungsteam kontaktieren möchtest, nutze bitte diesen [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU)-Kanal.

Der Qiskit Transpiler Service stellt Transpilierungsfunktionen in der Cloud bereit. Zusätzlich zu den lokalen Transpilierungsfunktionen des Qiskit Transpilers können deine Transpilierungsaufgaben von IBM Quantum Cloud-Ressourcen und KI-gestützten Transpiler-Passes profitieren.

Der Qiskit Transpiler Service bietet eine Python-Bibliothek, um diesen Service und seine Fähigkeiten nahtlos in deine bestehenden Qiskit-Muster und -Workflows zu integrieren. Dieser Service ist ausschließlich für Nutzer des IBM Quantum Premium Plan, Flex Plan und On-Prem (über die IBM Quantum Platform API) verfügbar.

<span id="install-transpiler-service"></span>
## Das Paket qiskit-ibm-transpiler installieren
Um den Qiskit Transpiler Service zu verwenden, installiere das Paket `qiskit-ibm-transpiler`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

Das Paket authentifiziert sich automatisch mit deinen [IBM Quantum Platform-Zugangsdaten](/guides/cloud-setup), entsprechend der Verwaltung durch [Qiskit Runtime](/guides/initialize-account):
- Umgebungsvariable: `QISKIT_IBM_TOKEN`
- Konfigurationsdatei `~/.qiskit/qiskit-ibm.json` (unter dem Abschnitt `default-ibm-quantum`).

*Hinweis*: Dieses Paket erfordert Qiskit SDK v1.X.

## Transpilierungsoptionen von qiskit-ibm-transpiler
- `backend_name` (optional, str) – Ein Backend-Name, wie er vom QiskitRuntimeService erwartet wird (zum Beispiel `ibm_torino`). Wenn dieser gesetzt ist, verwendet die Transpilierungsmethode das Layout des angegebenen Backends für die Transpilierung. Wenn eine andere Option gesetzt ist, die diese Einstellungen beeinflusst, zum Beispiel `coupling_map`, werden die `backend_name`-Einstellungen überschrieben.
- `coupling_map` (optional, List[List[int]]) – Eine gültige Coupling-Map-Liste (zum Beispiel [[0,1],[1,2]]). Wenn diese gesetzt ist, verwendet die Transpilierungsmethode diese Coupling Map für die Transpilierung. Wenn angegeben, überschreibt sie jeden für `target` angegebenen Wert.
- `optimization_level` (int) – Der Optimierungsgrad, der während der Transpilierung angewendet werden soll. Gültige Werte sind [1,2,3], wobei 1 die geringste Optimierung (und die schnellste) und 3 die stärkste Optimierung (und die zeitaufwändigste) bedeutet.
- `ai` ("true", "false", "auto") – Ob KI-gestützte Funktionen während der Transpilierung verwendet werden sollen. Die verfügbaren KI-gestützten Funktionen können `AIRouting`-Transpiler-Passes oder andere KI-gestützte Synthesemethoden umfassen. Bei `"true"` wendet der Service je nach angefordertem `optimization_level` verschiedene KI-gestützte Transpiler-Passes an. Bei `"false"` werden die neuesten Qiskit-Transpilierungsfunktionen ohne KI genutzt. Bei `"auto"` entscheidet der Service basierend auf deinem Circuit, ob die standardmäßigen Qiskit-Heuristik-Passes oder die KI-gestützten Passes angewendet werden.
- `qiskit_transpile_options` (dict) – Ein Python-Dictionary-Objekt, das beliebige andere Optionen enthalten kann, die in der [Qiskit-Methode `transpile()`](defaults-and-configuration-options) gültig sind. Wenn `qiskit_transpile_options` einen `optimization_level` enthält, wird dieser zugunsten des als Parameter angegebenen `optimization_level` verworfen. Enthält `qiskit_transpile_options` eine Option, die von der Qiskit-Methode `transpile()` nicht erkannt wird, gibt die Bibliothek einen Fehler aus.

Weitere Informationen zu den verfügbaren `qiskit-ibm-transpiler`-Methoden findest du in der [API-Referenz von qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler). Mehr über die Service-API erfährst du in der [REST-API-Dokumentation des Qiskit Transpiler Service](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest).

## Beispiele
Die folgenden Beispiele zeigen, wie Circuits mit dem Qiskit Transpiler Service und verschiedenen Parametern transpiliert werden.

1. Erstelle einen Circuit und rufe den Qiskit Transpiler Service auf, um den Circuit mit `ibm_torino` als `backend_name`, 3 als `optimization_level` und ohne KI-Einsatz während der Transpilierung zu transpilieren.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*Hinweis*: Du kannst als `backend_name` nur Geräte verwenden, auf die du mit deinem IBM Quantum-Konto Zugriff hast. Neben dem `backend_name` akzeptiert der `TranspilerService` auch `coupling_map` als Parameter.

2. Erstelle einen ähnlichen Circuit und transpiliere ihn, wobei du KI-gestützte Transpilierungsfunktionen anforderst, indem du das Flag `ai` auf `True` setzt:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. Erstelle einen ähnlichen Circuit und transpiliere ihn, wobei du den Service entscheiden lässt, ob die KI-gestützten Transpiler-Passes verwendet werden sollen.